# 03 — Robustness and critical evaluation

**Purpose:** test whether the conclusion survives reasonable alternative choices. Only run checks relevant to the chosen question.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.oecd_audit import INDICATOR_SPECS, load_clean

PEERS = ['CAN', 'NZL', 'GBR', 'USA']
CODES = ['1_1', '2_1', '2_7', '2_2', '1_2']

def scorecard(data, references, normal_only=False, codes=CODES, start=2010, end=2024):
    rows = []
    for code in codes:
        x = data.loc[data.indicator_code.eq(code) & data.country_code.isin(['AUS', *references]) & data.year.isin([start, end])].copy()
        if normal_only: x = x.loc[x.status_code.eq('A')]
        x = x.sort_values('year').drop_duplicates(['country_code', 'independent_period'], keep='last')
        w = x.pivot(index='country_code', columns='year', values='value').reindex(columns=[start, end]).dropna()
        if 'AUS' not in w.index or len(w) < 2: continue
        change = w[end] - w[start]; oriented = change * (1 if INDICATOR_SPECS[code].direction == 'higher' else -1)
        peer = oriented.drop('AUS'); rank = oriented.rank(ascending=False, method='average')['AUS']; n = len(oriented)
        rows.append({'indicator': data.loc[data.indicator_code.eq(code), 'indicator'].iloc[0], 'australia_oriented_change': oriented['AUS'], 'reference_median_oriented_change': peer.median(), 'australia_minus_reference_median': oriented['AUS']-peer.median(), 'australia_change_percentile': 100*(n-rank)/(n-1), 'reference_country_count': len(peer)})
    return pd.DataFrame(rows)

def leave_one_out(data):
    tables = []
    for omitted in PEERS:
        table = scorecard(data, [p for p in PEERS if p != omitted])
        table.insert(0, 'omitted_peer', omitted); tables.append(table)
    return pd.concat(tables, ignore_index=True)

df = load_clean()
all_references = sorted(set(df.country_code) - {'AUS'})

## Implemented checks

- **Reference population:** compare the English-speaking sensitivity group with every supplied country reporting common endpoints.
- **Data-quality flags:** repeat broad comparisons using only OECD-normal (`A`) endpoint observations.
- **Influential peer:** leave each English-speaking peer out in turn.
- **Timing:** retain only exact common endpoints; no imputation or latest-available substitution.

For each check, state what changed, why it is reasonable, the result, and whether the main conclusion changed.

In [ ]:
broad = scorecard(df, all_references)
normal_only = scorecard(df, all_references, normal_only=True)
leave_one_out = leave_one_out(df)

display(broad[['indicator', 'australia_oriented_change', 'reference_median_oriented_change',
               'australia_change_percentile', 'reference_country_count']])
display(normal_only[['indicator', 'australia_oriented_change', 'reference_median_oriented_change',
                     'australia_change_percentile', 'reference_country_count']])
display(leave_one_out[['omitted_peer', 'indicator', 'australia_minus_reference_median']])